In [8]:
from joblib import load
import pandas as pd
import geopandas as gpd
from geopandas import GeoDataFrame
import libpysal as lps
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point

In [9]:
ct=gpd.read_file('ct2023_predict.geojson')

In [10]:
ct['tree density']=ct['tree_count']/ct['area_hectare']

In [11]:
ct['good tree density']=ct['good_tree_count']/ct['area_hectare']

In [12]:
print(ct.columns)

Index(['shape_area', 'ntaname', 'cdtaname', 'shape_leng', 'boroname', 'ct2020',
       'nta2020', 'borocode', 'cdeligibil', 'geoid', 'boroct2020', 'cdta2020',
       'ctlabel', 'area_hectare', 'population_2023', 'pop_density',
       'unemployment', 'unrelated individuals', 'citizen ratio',
       'less than high school', 'GINI index', 'per capita income',
       'gross rent', 'tree_count', 'good_tree_count', 'lotarea_sum',
       'landuse_count', 'comarea_sum', 'resarea_sum', 'assesstot_sum',
       'numfloor_sum', 'bldgarea_sum', 'ComFAR', 'ResFAR', 'assesstot',
       'NumFloors', 'Avr floor area', 'building_count', 'building_height_sum',
       'avr height', 'building density', 'geometry', 'tree density',
       'good tree density'],
      dtype='object')


In [13]:
cols_to_convert = [
    'less than high school', 'citizen ratio', 'unemployment',
    'unrelated individuals', 'GINI index', 'gross rent','per capita income'
]

for col in cols_to_convert:
    ct[col] = pd.to_numeric(ct[col], errors='coerce')

## *Population Growth Classification*

In [14]:
forest1=load('predictive model/random_forest_growth.pkl')

In [15]:
ct['unrelated individual']=ct['unrelated individuals']

In [16]:
features_selected = ['less than high school', 'pop_density', 'citizen ratio', 'unemployment',
        'unrelated individual', 'GINI index', 'gross rent', 'assesstot','ResFAR',
        'building density', 'tree density', 'good tree density','per capita income']

In [17]:
ct[features_selected] = ct[features_selected].replace('', np.nan)

In [18]:
ct['pop_change_forest']=forest1.predict(ct[features_selected])

In [19]:
ct['pop_change_forest_prob_1'] = forest1.predict_proba(ct[features_selected])[:, 1]

In [26]:
category_counts = ct['pop_change_forest'].value_counts().sort_index()
print(category_counts)

pop_change_forest
0     640
1    1685
Name: count, dtype: int64


## *Classification of Population Growth Levels*

In [20]:
forest2=load('predictive model/forest_growth_type.pkl')

In [21]:
features_selected2 = ['less than high school', 'pop_density', 'citizen ratio', 'unemployment',
        'unrelated individual', 'GINI index', 'gross rent', 'assesstot','ComFAR',
        'building density', 'Avr floor area', 'good tree density','per capita income']

In [22]:
ct['pop_change_type_forest']=forest2.predict(ct[features_selected2])

In [23]:
ct['pop_change_type_forest_prob_2'] = forest2.predict_proba(ct[features_selected2])[..., 2]

In [24]:
ct['pop_change_type_forest_prob_0'] = forest2.predict_proba(ct[features_selected2])[..., 0]

In [ ]:
# ct.to_file('Result.geojson')

In [27]:
category_counts = ct['pop_change_type_forest'].value_counts().sort_index()
print(category_counts)

pop_change_type_forest
0     431
1    1236
2     658
Name: count, dtype: int64
